In [8]:
# General load and data prep 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')


df = pd.read_csv('WA_Fn-UseC_-HR-Employee-Attrition.csv')
df['Attrition_Flag'] = (df['Attrition'] == 'Yes').astype(int)
attrition_rate = df['Attrition_Flag'].mean()

# Tenure groups
df['TenureGroup'] = pd.cut(
    df['YearsAtCompany'],
    bins=[-1, 2, 5, 10, 100],
    labels=['0-2 yrs', '3-5 yrs', '6-10 yrs', '11+ yrs']
)

# Income quartiles (Q1 = lowest income, Q4 = highest)
df['IncomeQuartile'] = pd.qcut(df['MonthlyIncome'], 4, labels=['Q1', 'Q2', 'Q3', 'Q4'])

In [9]:
def profile_risk(mask, label, min_n=20):
    subset = df[mask]
    n = len(subset)
    rate = subset['Attrition_Flag'].mean() if n > 0 else np.nan
    flag = '' if n >= min_n else '  (small sample, n<20)'
    print(f'{label:45s} n={n:4d}   attrition rate={rate:.1%}{flag}')
    return n, rate

print('--- Candidate risk profiles ---')
profile_risk((df['OverTime']=='Yes') & (df['MaritalStatus']=='Single'), 'OverTime=Yes & Single')
profile_risk((df['OverTime']=='Yes') & (df['IncomeQuartile']=='Q1'), 'OverTime=Yes & Income Q1 (lowest)')
profile_risk((df['JobRole']=='Sales Representative') & (df['OverTime']=='Yes'), 'Sales Representative & OverTime=Yes')
profile_risk((df['OverTime']=='Yes') & (df['MaritalStatus']=='Single') & (df['IncomeQuartile']=='Q1'),
             'OverTime=Yes & Single & Income Q1')
profile_risk((df['TenureGroup']=='0-2 yrs') & (df['OverTime']=='Yes'), 'Tenure 0-2 yrs & OverTime=Yes')
profile_risk((df['JobRole']=='Laboratory Technician') & (df['OverTime']=='Yes'), 'Laboratory Technician & OverTime=Yes')
print()
print(f'(Company-wide baseline attrition rate: {attrition_rate:.1%})')


--- Candidate risk profiles ---
OverTime=Yes & Single                         n= 131   attrition rate=49.6%
OverTime=Yes & Income Q1 (lowest)             n= 106   attrition rate=58.5%
Sales Representative & OverTime=Yes           n=  24   attrition rate=66.7%
OverTime=Yes & Single & Income Q1             n=  41   attrition rate=70.7%
Tenure 0-2 yrs & OverTime=Yes                 n= 104   attrition rate=51.0%
Laboratory Technician & OverTime=Yes          n=  62   attrition rate=50.0%

(Company-wide baseline attrition rate: 16.1%)


In [10]:
# TOP 3 highest-risk profiles, ranked by attrition rate among segments with n>=20
risk_profiles = pd.DataFrame([
    {'profile': 'OverTime=Yes & Single & Income Q1 (lowest)', 'n': 41,
     'attrition_rate': df[(df['OverTime']=='Yes') & (df['MaritalStatus']=='Single') & (df['IncomeQuartile']=='Q1')]['Attrition_Flag'].mean()},
    {'profile': 'Sales Representative & OverTime=Yes', 'n': 24,
     'attrition_rate': df[(df['JobRole']=='Sales Representative') & (df['OverTime']=='Yes')]['Attrition_Flag'].mean()},
    {'profile': 'OverTime=Yes & Income Q1 (lowest)', 'n': 106,
     'attrition_rate': df[(df['OverTime']=='Yes') & (df['IncomeQuartile']=='Q1')]['Attrition_Flag'].mean()},
]).sort_values('attrition_rate', ascending=False).reset_index(drop=True)

risk_profiles['attrition_rate'] = (risk_profiles['attrition_rate'] * 100).round(1)
risk_profiles.index = risk_profiles.index + 1
risk_profiles.index.name = 'Rank'
risk_profiles.style.format({'attrition_rate': '{:.1f}%'}).background_gradient(cmap='Reds', subset=['attrition_rate'])

,profile,n,attrition_rate
Rank,,,
1,OverTime=Yes & Single & Income Q1 (lowest),41,70.7%
2,Sales Representative & OverTime=Yes,24,66.7%
3,OverTime=Yes & Income Q1 (lowest),106,58.5%


### 6 Key Findings

1. **Overall attrition is 16.1%** (237 of 1,470 employees), but risk is highly concentrated rather than evenly spread — several segments run 2–4x the company average.
2. **OverTime is the single strongest behavioral driver**: employees working overtime leave at 30.5% vs. 10.4% for those who don't — nearly a 3x gap.
3. **New hires are the most fragile population**: employees with 0–2 years of tenure churn at 29.8%, more than triple the 8.1% rate for 11+-year veterans.
4. **Compounding risk is severe**: employees who are simultaneously on overtime, single, and in the lowest income quartile churn at 70.7% — over 4x the company baseline — versus 41 people in that exact segment.
5. **Sales Representatives are the highest-risk job role** at 39.8% attrition (vs. 2.5% for Research Directors), and that risk climbs to 66.7% when combined with overtime.
6. **Low job satisfaction and poor work-life balance compound within Sales and HR**: Sales employees with JobSatisfaction=1 churn at 26.7%, and HR employees with WorkLifeBalance=1 show 0% in this dataset only due to a very small sample — the Sales and R&D low-satisfaction/low-WLB cells are the ones with enough volume to act on.

### 5 HR Recommendations

1. **Audit and cap overtime for Sales Representatives and Laboratory Technicians** — these two roles combine high baseline attrition with the overtime multiplier; workload redistribution or overtime pay/policy review is the highest-leverage single intervention available.
2. **Build a structured 0–2 year onboarding and retention program** — with new hires churning at nearly 30%, front-loaded mentorship, check-ins, and early career-pathing could meaningfully blunt the steepest part of the attrition curve.
3. **Review compensation for the lowest income quartile, especially where it intersects with overtime** — Q1 earners churn at 29.3% company-wide and 58.5% when also working overtime; targeted pay adjustments for this group would directly address the largest compounding risk factor found.
4. **Create a proactive retention watchlist for the "triple-risk" profile** (overtime + single + Q1 income) — with a 70.7% attrition rate in this exact segment, even a small, well-resourced retention outreach to these ~40 employees could meaningfully move the needle on total attrition numbers.
5. **Prioritize job-satisfaction and work-life-balance interventions in Sales** — Sales shows the highest attrition rate of any department (20.6%) and its lowest-satisfaction, lowest-WLB employees churn well above department average; manager training and workload review in Sales should be sequenced ahead of Department/R&D given where the volume of at-risk employees actually sits.